In [1]:
import pye57
from pathlib import Path
from pathlib import Path
import numpy as np
import pye57
import open3d as o3d

In [2]:
# Initialize an empty E57 reader object
# (Replace with a path to a small sample file when you get one from the techs)
try:
    e57_path = "data/test_cloud.e57"
    # This just tests if the underlying C++ extension initializes safely
    print("pye57 imported successfully!")
except Exception as e:
    print(f"Error: {e}")

pye57 imported successfully!


In [4]:
# 1. Path resolution
repo_root = Path.cwd().parent
e57_path = str(repo_root / "data" / "test_cloud.e57")

# 2. Open file
e57 = pye57.E57(e57_path)

# 3. Read scan 0 with strict missing-field tolerances
# Using the parameter suggested by the ValueError
data = e57.read_scan(0, intensity=True, colors=True, ignore_missing_fields=True)

# 4. Check what fields actually successfully loaded
print("Successfully extracted fields from your E57:")
for key in data.keys():
    print(f" - {key}")

Successfully extracted fields from your E57:
 - cartesianX
 - cartesianY
 - cartesianZ
 - intensity
 - colorRed
 - colorGreen
 - colorBlue


In [5]:
# 1. Extract raw coordinates
x = data["cartesianX"]
y = data["cartesianY"]
z = data["cartesianZ"]

# 2. Combine into an (N, 3) matrix
points_3d = np.column_stack((x, y, z))
print(f"Spatial matrix shape: {points_3d.shape}")
print(f"Total points ready for processing: {points_3d.shape[0]:,}")

# 3. Grab the scanner's transformation matrix
scanner_pose = e57.scan_position(0)
print("\nScanner Position Matrix (Pose):")
print(scanner_pose)

Spatial matrix shape: (11038828, 3)
Total points ready for processing: 11,038,828

Scanner Position Matrix (Pose):
[[70.72577256 81.18256614 31.588985  ]]


In [6]:
# 1. Initialize the Open3D PointCloud object
pcd = o3d.geometry.PointCloud()

# 2. Load your NumPy points into Open3D format
pcd.points = o3d.utility.Vector3dVector(points_3d)

# 3. Voxel downsample to 5cm chunks
pcd_down = pcd.voxel_down_sample(voxel_size=0.05)

print(f"Original point cloud size: {len(pcd.points):,}")
print(f"Downsampled point cloud size: {len(pcd_down.points):,}")

Original point cloud size: 11,038,828
Downsampled point cloud size: 301,510


In [7]:
# 1. Open the interactive desktop window
print("Launching Open3D visualizer window...")
print("-> Click and drag to rotate the cloud.")
print("-> Press 'q' on your keyboard to close the window and return to the notebook.")

o3d.visualization.draw_geometries([pcd_down])
print("Visualizer closed successfully.")

Launching Open3D visualizer window...
-> Click and drag to rotate the cloud.
-> Press 'q' on your keyboard to close the window and return to the notebook.
Visualizer closed successfully.
